# 🧬 DINOv2 Skin Cancer Classification — Google Colab Training

Fine-tune DINOv2 (facebook/dinov2-base) on ISIC 2019 dataset

**5 Classes:** MEL, BCC, SCC, DF, NV

### 📋 วิธีใช้บน Google Colab:
1. เปิด Notebook นี้บน Google Colab
2. ไปที่ **Runtime → Change runtime type → GPU (T4)**
3. อัปโหลดไฟล์ `kaggle.json` (Kaggle API Key) เมื่อถูกถาม
4. Run All → รอเทรนเสร็จ
5. โมเดลจะถูก save ไว้ใน Google Drive

### 📊 5 คลาสที่เทรน:
| Index | Class | Description |
|-------|-------|-------------|
| 0 | melanoma | เมลาโนมา (มะเร็งร้ายแรงที่สุด) |
| 1 | basal_cell_carcinoma | มะเร็งเซลล์ฐาน (BCC) |
| 2 | squamous_cell_carcinoma | มะเร็งเซลล์สความัส (SCC) ← ตัวจริง! |
| 3 | dermatofibroma | เนื้องอกไม่ร้าย |
| 4 | melanocytic_Nevi | ไฝ |

## 📦 CELL 1: Install Dependencies & Setup Colab

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn safetensors pillow kaggle

import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

print("=" * 65)
print("  🧬 DINOv2 Skin Cancer Classifier — Google Colab Training")
print("  📋 5 Classes | ISIC 2019 Dataset (มี SCC จริง!) | Colab GPU")
print("=" * 65)

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"  🖥️  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("  ⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU")
print()

## 📦 CELL 2: Mount Google Drive & Download Dataset

In [ ]:
# ── Mount Google Drive (สำหรับ save โมเดล) ──
from google.colab import drive
drive.mount('/content/drive')

# ── Setup Kaggle API ──
kaggle_json_path = Path('/content/drive/MyDrive/kaggle.json')

if kaggle_json_path.exists():
    print("  ✅ Found kaggle.json in Google Drive")
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    !cp "{kaggle_json_path}" ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
else:
    print("  📤 กรุณาอัปโหลดไฟล์ kaggle.json")
    print("  📋 วิธีได้มา: https://www.kaggle.com/settings → API → Create New Token")
    from google.colab import files
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    !mv kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json

print("  ✅ Kaggle API configured!")

# ── Download ISIC 2019 Dataset ──
DATA_DIR = Path('/content/isic2019')

if not DATA_DIR.exists():
    print("\n  📥 Downloading ISIC 2019 dataset from Kaggle...")
    print("  ⏳ This may take 5-10 minutes...")
    
    # ลอง dataset slug หลายตัว (บางตัวอาจถูกลบ/เปลี่ยนชื่อ)
    import subprocess
    dataset_slugs = [
        'nroman/2019-isic-challenge',
        'andrewmvd/isic-2019',
        'salviohexia/isic-2019-skin-lesion-images-for-classification',
    ]
    
    download_success = False
    for slug in dataset_slugs:
        print(f"  🔄 Trying: {slug}")
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', slug, '-p', '/content/isic2019', '--unzip'],
            capture_output=True, text=True
        )
        if result.returncode == 0 and DATA_DIR.exists() and any(DATA_DIR.iterdir()):
            print(f"  ✅ Download complete from: {slug}")
            download_success = True
            break
        else:
            err_msg = (result.stderr or result.stdout or 'Unknown error').strip()[:120]
            print(f"  ⚠️ Failed: {err_msg}")
    
    # ถ้าโหลดไม่ได้เลย → ให้ผู้ใช้อัปโหลดเอง
    if not download_success:
        print("\n  ❌ ไม่สามารถดาวน์โหลด dataset อัตโนมัติได้")
        print("  📋 กรุณาทำตามขั้นตอน:")
        print("     1. ไปที่ https://www.kaggle.com/datasets/nroman/2019-isic-challenge")
        print("     2. กด 'Download' (อาจต้อง Accept Rules ก่อน)")
        print("     3. อัปโหลดไฟล์ zip ที่ได้ขึ้นมา")
        print("")
        from google.colab import files
        uploaded = files.upload()
        
        # Unzip uploaded file
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        for fname in uploaded.keys():
            if fname.endswith('.zip'):
                !unzip -q -o "/content/{fname}" -d /content/isic2019
                print(f"  ✅ Extracted: {fname}")
else:
    print(f"  ✅ Dataset already exists at {DATA_DIR}")

# Show contents (with safety check)
if DATA_DIR.exists():
    print(f"\n  📁 Dataset contents:")
    for item in sorted(DATA_DIR.iterdir()):
        if item.is_dir():
            count = len(list(item.iterdir()))
            print(f"     📂 {item.name}/ ({count} files)")
        else:
            size_mb = item.stat().st_size / 1e6
            print(f"     📄 {item.name} ({size_mb:.1f} MB)")
else:
    print("  ❌ Dataset directory not found! กรุณารัน Cell นี้ใหม่")

## 📦 CELL 3: Configuration

In [ ]:
class Config:
    # ── Model ──
    MODEL_NAME = "facebook/dinov2-base"          # DINOv2 Base (ViT-B/14)
    NUM_CLASSES = 5
    
    # ── Training Hyperparameters ──
    LEARNING_RATE = 2e-5
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 16
    SEED = 42
    NUM_EPOCHS = 15
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    
    # ── Data ──
    IMAGE_SIZE = 224
    
    # ── ISIC 2019: 5 คลาสที่ต้องการ ──
    CLASS_NAMES = [
        "melanoma",                       # MEL  — เมลาโนมา
        "basal_cell_carcinoma",           # BCC  — มะเร็งเซลล์ฐาน
        "squamous_cell_carcinoma",        # SCC  — มะเร็งเซลล์สความัส (ตัวจริง!)
        "dermatofibroma",                 # DF   — เนื้องอกไม่ร้าย
        "melanocytic_Nevi",               # NV   — ไฝ
    ]
    
    # ISIC 2019 label column names → our index
    ISIC_TO_IDX = {
        "MEL":  0,   # melanoma
        "BCC":  1,   # basal_cell_carcinoma
        "SCC":  2,   # squamous_cell_carcinoma  ← SCC จริง!
        "DF":   3,   # dermatofibroma
        "NV":   4,   # melanocytic_Nevi
    }
    
    # Classes to KEEP from ISIC 2019
    KEEP_CLASSES = ["MEL", "BCC", "SCC", "DF", "NV"]
    
    # ── Paths (Google Colab) ──
    COLAB_DATA = Path("/content/isic2019")
    OUTPUT_DIR = Path("/content/drive/MyDrive/dinov2-skin-cancer-5class")
    
    # ── Class Weights for Imbalanced Data ──
    USE_CLASS_WEIGHTS = True


config = Config()

# Set seed for reproducibility
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

print(f"  📋 Model: {config.MODEL_NAME}")
print(f"  🎯 Classes: {config.NUM_CLASSES}")
for i, name in enumerate(config.CLASS_NAMES):
    print(f"     [{i}] {name}")
print(f"  📊 Epochs: {config.NUM_EPOCHS}")
print(f"  📊 LR: {config.LEARNING_RATE}")
print(f"  📊 Batch: {config.TRAIN_BATCH_SIZE}")
print(f"  💾 Output: {config.OUTPUT_DIR}")
print()

## 📦 CELL 4: Load & Prepare ISIC 2019 Dataset

In [ ]:
print("📂 Loading ISIC 2019 dataset...")

# ── Find the ground truth CSV ──
import glob
gt_path = None

# ค้นหา CSV ทุกที่ใน /content/isic2019
all_csvs = glob.glob("/content/isic2019/**/*.csv", recursive=True)
print(f"  🔍 Found CSVs: {all_csvs[:10]}")

# ลำดับความสำคัญในการค้นหา
for csv in all_csvs:
    csv_lower = csv.lower()
    if 'groundtruth' in csv_lower or 'ground_truth' in csv_lower:
        gt_path = Path(csv)
        break

if gt_path is None:
    for csv in all_csvs:
        if 'training' in csv.lower():
            gt_path = Path(csv)
            break

if gt_path is None and all_csvs:
    gt_path = Path(all_csvs[0])

assert gt_path is not None, """
❌ Cannot find ISIC 2019 Ground Truth CSV!
Please check:
  1. Kaggle API key is correct
  2. Dataset downloaded successfully
  3. Re-run CELL 2 to download dataset
"""
print(f"  ✅ Ground Truth: {gt_path}")

# Load CSV
df = pd.read_csv(gt_path)
print(f"  📊 Total images: {len(df)}")
print(f"  📊 Columns: {list(df.columns)}")

# ── Convert one-hot to single label ──
label_columns = [col for col in df.columns if col.upper() in ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']]

if label_columns:
    print(f"  📊 Label columns found: {label_columns}")
    df['dx'] = df[label_columns].idxmax(axis=1).str.upper()
else:
    possible_label_cols = ['diagnosis', 'dx', 'label', 'class']
    for col in possible_label_cols:
        if col in df.columns:
            df['dx'] = df[col].str.upper()
            break

print(f"  📊 Original class distribution:")
print(df['dx'].value_counts().to_string())

# ── Filter: keep only our 5 classes ──
df = df[df['dx'].isin(config.KEEP_CLASSES)].copy()
df['label'] = df['dx'].map(config.ISIC_TO_IDX)
print(f"\n  📊 After filtering (5 classes): {len(df)}")
print(f"  📊 Selected class distribution:")
for dx in config.KEEP_CLASSES:
    count = (df['dx'] == dx).sum()
    idx = config.ISIC_TO_IDX[dx]
    print(f"     [{idx}] {config.CLASS_NAMES[idx]:35s} ({dx:4s}): {count:6d}")

# ── Find image files ──
image_name_col = 'image' if 'image' in df.columns else df.columns[0]
print(f"\n  📊 Image name column: '{image_name_col}'")

# Search for image directories everywhere under /content/isic2019
import glob as glob_module
image_dirs = []

# ค้นหาโฟลเดอร์ที่มีรูปภาพ
for pattern in ['/content/isic2019/**/ISIC_2019_Training_Input',
                '/content/isic2019/**/Train',
                '/content/isic2019/**/train',
                '/content/isic2019/**/images']:
    for d in glob_module.glob(pattern, recursive=True):
        image_dirs.append(Path(d))

# เพิ่ม root dir เผื่อรูปอยู่ตรงนั้น
image_dirs.append(config.COLAB_DATA)

# เพิ่มทุก subdirectory ที่มีไฟล์ jpg
for jpg in glob_module.glob('/content/isic2019/**/*.jpg', recursive=True):
    parent = Path(jpg).parent
    if parent not in image_dirs:
        image_dirs.append(parent)
    break  # แค่หาโฟลเดอร์แรกที่มีรูป

image_dirs = list(set(image_dirs))
print(f"  📊 Image directories: {image_dirs}")

# Build image_name → file path mapping
image_paths = {}
for d in image_dirs:
    if not d.exists():
        continue
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        for img_file in d.glob(ext):
            image_paths[img_file.stem] = str(img_file)

print(f"  📊 Found {len(image_paths)} image files")

# Add file path to dataframe
df['filepath'] = df[image_name_col].map(image_paths)

# If not matched, try alternative matching
if df['filepath'].isna().sum() > len(df) * 0.5:
    print(f"  ⚠️ Low match rate, trying alternative matching...")
    for idx, row in df.iterrows():
        name = str(row[image_name_col])
        if name in image_paths:
            df.at[idx, 'filepath'] = image_paths[name]
        else:
            for variant in [name, f"ISIC_{name}", name.replace("ISIC_", "")]:
                if variant in image_paths:
                    df.at[idx, 'filepath'] = image_paths[variant]
                    break

df = df.dropna(subset=['filepath'])
print(f"  📊 Matched images: {len(df)}")

assert len(df) > 100, f"❌ Only matched {len(df)} images! Check dataset path."

# ── Train/Val split (stratified) ──
train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=config.SEED, stratify=df['label']
)
print(f"\n  📊 Train: {len(train_df)} | Val: {len(val_df)}")

# Show per-class split
print(f"\n  📊 Per-class split:")
for dx in config.KEEP_CLASSES:
    idx = config.ISIC_TO_IDX[dx]
    train_count = (train_df['dx'] == dx).sum()
    val_count = (val_df['dx'] == dx).sum()
    print(f"     [{idx}] {config.CLASS_NAMES[idx]:35s}: Train={train_count:5d} | Val={val_count:4d}")
print()

## 📦 CELL 5: Dataset & Transforms

In [ ]:
# DINOv2 normalization (ImageNet stats)
DINOV2_MEAN = [0.485, 0.456, 0.406]
DINOV2_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])


class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        label = int(row['label'])
        
        if self.transform:
            image = self.transform(image)
        
        return {"pixel_values": image, "labels": label}


train_dataset = SkinCancerDataset(train_df, transform=train_transforms)
val_dataset = SkinCancerDataset(val_df, transform=val_transforms)

print(f"  ✅ Train dataset: {len(train_dataset)} samples")
print(f"  ✅ Val dataset: {len(val_dataset)} samples")

# Compute class weights for imbalanced data
if config.USE_CLASS_WEIGHTS:
    class_counts = train_df['label'].value_counts().sort_index().values
    total = class_counts.sum()
    class_weights = total / (config.NUM_CLASSES * class_counts)
    class_weights = torch.FloatTensor(class_weights).to(device)
    print(f"\n  ⚖️ Class weights (inverse frequency):")
    for i, (name, w) in enumerate(zip(config.CLASS_NAMES, class_weights)):
        print(f"     [{i}] {name:35s}: {w:.3f}")
else:
    class_weights = None

print()

## 📦 CELL 6: Build DINOv2 Model

In [ ]:
from transformers import Dinov2Model, Dinov2PreTrainedModel, Dinov2Config
from transformers.modeling_outputs import ImageClassifierOutput
import torch.nn as nn


class Dinov2ForSkinCancer(Dinov2PreTrainedModel):
    """
    DINOv2 with classification head for skin cancer detection.
    Uses the [CLS] token output + linear classifier.
    """
    def __init__(self, config):
        super().__init__(config)
        self.dinov2 = Dinov2Model(config)
        self.classifier = nn.Sequential(
            nn.LayerNorm(config.hidden_size),
            nn.Dropout(p=0.1),
            nn.Linear(config.hidden_size, 512),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(512, config.num_labels),
        )
        self.num_labels = config.num_labels
        self.post_init()
    
    def forward(self, pixel_values, labels=None):
        outputs = self.dinov2(pixel_values)
        # Use [CLS] token (first token)
        cls_output = outputs.last_hidden_state[:, 0]
        logits = self.classifier(cls_output)
        
        loss = None
        if labels is not None:
            if class_weights is not None:
                loss_fn = nn.CrossEntropyLoss(weight=class_weights)
            else:
                loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        
        return ImageClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


print("🧠 Loading DINOv2 model...")

# Load DINOv2 config and modify for our task
dinov2_config = Dinov2Config.from_pretrained(config.MODEL_NAME)
dinov2_config.num_labels = config.NUM_CLASSES
dinov2_config.id2label = {i: name for i, name in enumerate(config.CLASS_NAMES)}
dinov2_config.label2id = {name: i for i, name in enumerate(config.CLASS_NAMES)}

# Create model with pre-trained DINOv2 backbone
model = Dinov2ForSkinCancer(dinov2_config)

# Load pre-trained DINOv2 weights (backbone only)
pretrained = Dinov2Model.from_pretrained(config.MODEL_NAME)
model.dinov2.load_state_dict(pretrained.state_dict())
del pretrained

# Freeze early layers (optional, for faster training)
for param in model.dinov2.embeddings.parameters():
    param.requires_grad = False

for i, layer in enumerate(model.dinov2.encoder.layer):
    if i < 8:  # Freeze first 8 of 12 layers
        for param in layer.parameters():
            param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  ✅ Model loaded!")
print(f"  📊 Total params: {total_params:,}")
print(f"  📊 Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"  📊 Frozen params: {total_params - trainable_params:,}")

model = model.to(device)

# Free up memory
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print()

## 📦 CELL 7: Training Setup

In [ ]:
from torch.optim import AdamW
import math

# DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.TRAIN_BATCH_SIZE, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.EVAL_BATCH_SIZE, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True,
)

# Optimizer: AdamW
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

# Cosine LR Scheduler with warmup
total_steps = len(train_loader) * config.NUM_EPOCHS
warmup_steps = int(total_steps * config.WARMUP_RATIO)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed precision (Native AMP)
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

print(f"  📊 Total training steps: {total_steps}")
print(f"  📊 Warmup steps: {warmup_steps}")
print(f"  📊 Steps per epoch: {len(train_loader)}")
print()

## 📦 CELL 8: Training Loop

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def train_one_epoch(model, loader, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch_idx, batch in enumerate(loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.amp.autocast("cuda"):
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        scheduler.step()
        
        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        if (batch_idx + 1) % 10 == 0:
            lr = scheduler.get_last_lr()[0]
            print(f"    Step {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f} | LR: {lr:.2e}")
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        
        if device.type == "cuda":
            with torch.amp.autocast("cuda"):
                outputs = model(pixel_values=pixel_values, labels=labels)
        else:
            outputs = model(pixel_values=pixel_values, labels=labels)
        
        total_loss += outputs.loss.item()
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, acc, f1, np.array(all_preds), np.array(all_labels)


# ── Start Training ──
print("=" * 65)
print("  🚀 STARTING TRAINING — DINOv2 + ISIC 2019 (5 classes + SCC)")
print("=" * 65)

best_f1 = 0.0
best_epoch = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'─' * 65}")
    print(f"  📊 Epoch {epoch+1}/{config.NUM_EPOCHS}")
    print(f"{'─' * 65}")
    
    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, device, epoch
    )
    
    # Evaluate 
    val_loss, val_acc, val_f1, val_preds, val_labels = evaluate(model, val_loader, device)
    
    # Log
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    
    print(f"\n  📈 Results:")
    print(f"     Train — Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"     Val   — Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    
    # Save best model to Google Drive
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch + 1
        
        config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(config.OUTPUT_DIR)
        print(f"     💾 Best model saved to Google Drive! (F1: {best_f1:.4f})")
    
    print(f"     🏆 Best: Epoch {best_epoch} (F1: {best_f1:.4f})")

print(f"\n{'=' * 65}")
print(f"  ✅ TRAINING COMPLETE!")
print(f"  🏆 Best Epoch: {best_epoch} | Best F1: {best_f1:.4f}")
print(f"{'=' * 65}")

## 📦 CELL 9: Final Evaluation & Classification Report

In [ ]:
print("\n📋 Loading best model for final evaluation...")

# Load best model
best_model = Dinov2ForSkinCancer.from_pretrained(config.OUTPUT_DIR)
best_model = best_model.to(device)

# Final evaluation
val_loss, val_acc, val_f1, val_preds, val_labels = evaluate(best_model, val_loader, device)

print(f"\n{'=' * 65}")
print(f"  📊 FINAL CLASSIFICATION REPORT")
print(f"{'=' * 65}")
print(classification_report(
    val_labels, val_preds, 
    target_names=config.CLASS_NAMES,
    digits=4
))

# Confusion Matrix
print(f"\n  📊 Confusion Matrix:")
cm = confusion_matrix(val_labels, val_preds)
print(pd.DataFrame(
    cm, 
    index=[f"True:{n}" for n in config.CLASS_NAMES],
    columns=[f"Pred:{n}" for n in config.CLASS_NAMES]
))

## 📦 CELL 10: Save for Production

In [ ]:
print(f"\n{'=' * 65}")
print(f"  💾 SAVING FOR PRODUCTION")
print(f"{'=' * 65}")

import json

production_config = {
    "architectures": ["Dinov2ForImageClassification"],
    "_name_or_path": "facebook/dinov2-base",
    "model_type": "dinov2",
    "num_labels": config.NUM_CLASSES,
    "id2label": {str(i): name for i, name in enumerate(config.CLASS_NAMES)},
    "label2id": {name: i for i, name in enumerate(config.CLASS_NAMES)},
    "image_size": config.IMAGE_SIZE,
    "hidden_size": 768,
    "num_attention_heads": 12,
    "num_hidden_layers": 12,
    "patch_size": 14,
    "torch_dtype": "float32",
    "training_info": {
        "dataset": "ISIC 2019 (25,331 images)",
        "num_classes": config.NUM_CLASSES,
        "classes": config.CLASS_NAMES,
        "epochs": config.NUM_EPOCHS,
        "learning_rate": config.LEARNING_RATE,
        "batch_size": config.TRAIN_BATCH_SIZE,
        "best_epoch": best_epoch,
        "best_val_f1": float(best_f1),
        "best_val_acc": float(val_acc),
        "optimizer": "AdamW (betas=0.9,0.999, eps=1e-8)",
        "lr_scheduler": "cosine_with_warmup (ratio=0.1)",
        "augmentation": "flip+rotate+colorjitter+affine",
        "class_weights": "inverse_frequency",
        "frozen_layers": "embeddings + first 8/12 transformer layers",
        "mixed_precision": "Native AMP (fp16)",
        "platform": "Google Colab",
    }
}

config_path = config.OUTPUT_DIR / "training_config.json"
with open(config_path, 'w') as f:
    json.dump(production_config, f, indent=2, ensure_ascii=False)

print(f"  ✅ Model saved to Google Drive: {config.OUTPUT_DIR}")
print(f"  📁 Files:")
for f in sorted(config.OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"     {f.name:30s} ({size_mb:.1f} MB)")

print(f"\n  📋 Class mapping (ใช้ในโปรเจค):")
for i, name in enumerate(config.CLASS_NAMES):
    print(f"     [{i}] {name}")

print(f"\n{'=' * 65}")
print(f"  ✅ ALL DONE! Model files saved in Google Drive:")
print(f"  📁 {config.OUTPUT_DIR}")
print(f"  📁 Copy to your project's model folder:")
print(f"     - model.safetensors")
print(f"     - config.json")
print(f"{'=' * 65}")

## 📦 CELL 11: Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', color='#00647c', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', color='#007f9d', linewidth=2, linestyle='--')
axes[0].set_title('Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', color='#00647c', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', color='#007f9d', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train', color='#00647c', linewidth=2)
axes[2].plot(history['val_f1'], label='Val', color='#007f9d', linewidth=2, linestyle='--')
axes[2].set_title('F1 Score (Macro)', fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('DINOv2 Skin Cancer (5 Classes + SCC) — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("  📊 Training plot saved to Google Drive!")

## 📦 CELL 12: Download Model (Optional)

ถ้าต้องการดาวน์โหลดโมเดลลงเครื่อง (นอกจาก Google Drive)

In [ ]:
# ── Optional: Download model files to local machine ──
from google.colab import files

print("📥 Preparing model files for download...")

# Zip the model folder for easy download
import shutil
zip_path = '/content/dinov2-skin-cancer-5class'
shutil.make_archive(zip_path, 'zip', config.OUTPUT_DIR)

print(f"  ✅ Model zipped: {zip_path}.zip")
print(f"  📥 Starting download...")

files.download(f'{zip_path}.zip')
print("  ✅ Download started!")